# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [102]:
import importlib.util
from pathlib import Path

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [103]:
# Parâmetros importados do arquivo config.py
print(
    "Filtrar por quantidade de alelos TCC1:.........................",
    parametros.filtarar_por_qte_de_alelos_tcc1,
)
print(
    "Parâmetro de filtragem median binding percentile TCC1:.........",
    parametros.parametro_de_filtragem_mbp_tcc1,
)
print(
    "Percentual de match mínimo TCC1:...............................",
    parametros.percent_match_minimo_tcc1,
)

Filtrar por quantidade de alelos TCC1:......................... 10
Parâmetro de filtragem median binding percentile TCC1:......... 1
Percentual de match mínimo TCC1:............................... 95.0


In [104]:
import pandas as pd

In [105]:
df = pd.read_csv('./T CELL/DENV 1 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTGKTIHEW,303,311,9,HLA-B*57:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.997406,0.01
1,1,VTGKTIHEW,303,311,9,HLA-B*58:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.996622,0.01
2,1,SEMIIPKIY,239,247,9,HLA-B*44:03,239,0.01,SEMIIPKIY,SEMIIPKIY,0.996239,0.01
3,1,SEMIIPKIY,239,247,9,HLA-B*44:02,239,0.01,SEMIIPKIY,SEMIIPKIY,0.992169,0.01
4,1,LSAAIGKAW,42,50,9,HLA-B*57:01,42,0.01,LSAAIGKAW,LSAAIGKAW,0.989554,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,PVKEKEENLVKS,337,348,12,HLA-B*53:01,1366,100.00,PVEENLVKS,PVKEKEENLVKS,0.000000,100.00
36986,1,EKEENLVKSMVS,340,351,12,HLA-A*11:01,1369,100.00,ENLVKSMVS,EKEENLVKSMVS,0.000000,100.00
36987,1,EKEENLVKSMVS,340,351,12,HLA-A*24:02,1369,100.00,EKEENKSMV,EKEENLVKSMV,0.000000,100.00
36988,1,EKEENLVKSMVS,340,351,12,HLA-A*32:01,1369,100.00,ENLVKSMVS,EKEENLVKSMVS,0.000000,100.00


## Selecionando Epítopos por median binding percentile.

In [106]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc1

In [107]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  1


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTGKTIHEW,303,311,9,HLA-B*57:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.997406,0.01
1,1,VTGKTIHEW,303,311,9,HLA-B*58:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.996622,0.01
2,1,SEMIIPKIY,239,247,9,HLA-B*44:03,239,0.01,SEMIIPKIY,SEMIIPKIY,0.996239,0.01
3,1,SEMIIPKIY,239,247,9,HLA-B*44:02,239,0.01,SEMIIPKIY,SEMIIPKIY,0.992169,0.01
4,1,LSAAIGKAW,42,50,9,HLA-B*57:01,42,0.01,LSAAIGKAW,LSAAIGKAW,0.989554,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
549,1,IWPKSHTLW,224,232,9,HLA-B*53:01,224,0.97,IWPKSHTLW,IWPKSHTLW,0.073938,0.97
550,1,VVVDEHCGNR,285,294,10,HLA-A*68:01,629,0.98,VVVDEHCNR,VVVDEHCGNR,0.346667,0.98
551,1,KAVHADMGYWI,192,202,11,HLA-B*58:01,879,0.98,KAVHADMYW,KAVHADMGYW,0.148467,0.98
552,1,ILLENDMKF,78,86,9,HLA-B*35:01,78,0.98,ILLENDMKF,ILLENDMKF,0.107101,0.98


## Agrupando por pepitideos e agregando colunas pertinentes

In [108]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ADSPKRLSAAI,36,46,1,0.510,HLA-B*07:02
1,ASFIEVKTCIW,215,225,3,0.120,"HLA-A*32:01, HLA-B*57:01, HLA-B*58:01"
2,ATRLENIMW,60,68,4,0.105,"HLA-A*32:01, HLA-B*53:01, HLA-B*57:01, HLA-B*5..."
3,ATRLENIMWK,60,69,4,0.210,"HLA-A*03:01, HLA-A*11:01, HLA-A*30:01, HLA-B*5..."
4,AVHADMGYW,193,201,5,0.270,"HLA-A*26:01, HLA-A*32:01, HLA-B*53:01, HLA-B*5..."
...,...,...,...,...,...,...
219,YGFGIFTTNIW,158,168,3,0.390,"HLA-B*53:01, HLA-B*57:01, HLA-B*58:01"
220,YGGPISQHNY,247,256,2,0.470,"HLA-A*01:01, HLA-A*30:02"
221,YGGPISQHNYR,247,257,1,0.880,HLA-A*33:01
222,YGMEIRPVK,331,339,1,0.400,HLA-A*30:01


## Filtragem por qte_de_alelos

In [109]:
# mbp_minimo = 1
qte_alelos_minimo = parametros.filtarar_por_qte_de_alelos_tcc1

In [110]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= qte_alelos_minimo
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,HTWTEQYKF,26,34,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,QPMEHKYSW,107,115,10,0.405,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3..."


## Sorting por median_biding_percentile

In [111]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,HTWTEQYKF,26,34,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,QPMEHKYSW,107,115,10,0.405,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [112]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_1.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0    HTWTEQYKF
1    QPMEHKYSW
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [113]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [114]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_1.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,HTWTEQYKF,9,99.80% (1013/1015),88.89%,100.00%,NaN
1,2,NP 2,QPMEHKYSW,9,63.65% (646/1015),77.78%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [115]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# MPB
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "median_binding_percentile"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# start
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "start"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# end
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "end"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos
0,NP 1,HTWTEQYKF,9,99.80% (1013/1015),88.89%,100.00%,10,0.280,26,34,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,NP 2,QPMEHKYSW,9,63.65% (646/1015),77.78%,100.00%,10,0.405,107,115,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [116]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 1,HTWTEQYKF,9,99.80% (1013/1015),88.89%,100.00%,10,0.280,26,34,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",99.80
1,NP 2,QPMEHKYSW,9,63.65% (646/1015),77.78%,100.00%,10,0.405,107,115,"HLA-A*32:01, HLA-B*07:02, HLA-B*08:01, HLA-B*3...",63.65


### Sort e filtragem por percent_match e presença em alelos

In [117]:
minimum_percent_match = parametros.percent_match_minimo_tcc1

In [118]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= minimum_percent_match]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 1,HTWTEQYKF,9,99.80% (1013/1015),88.89%,100.00%,10,0.28,26,34,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",99.8


### Estrutura secundária

In [119]:
def processar_horiz(caminho_horiz):
    """
    Lê o arquivo .horiz do PSIPRED e retorna um dicionário 
    contendo as strings de confiança ('conf'), predição ('pred') e sequência ('seq').
    """
    confianca = ""
    predicao = ""
    sequencia = ""
    
    with open(caminho_horiz, 'r') as f:
        for linha in f:
            linha_str = linha.strip()
            if linha_str.startswith("Conf:"):
                confianca += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("Pred:"):
                predicao += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("AA:"):
                sequencia += linha_str.split(":", 1)[1].strip()
                
    return {
        "conf": confianca,
        "pred": predicao,
        "seq": sequencia
    }

In [120]:
denv_horiz = processar_horiz("./PSIPRED/denv.horiz")
denv_horiz

{'conf': '9013886999939925069997554001021479749988999999999973938999698999999996388999999877985799988722046145131014898520153204401023577789479980899987765453897799999567887505999866444555892562167009978874264488861289479999999643764379412568888631112058565788543446899641358893138189988317999799789889997531346269961362630579999869995994061134671687656437855329',
 'pred': 'CCCCEEECCCCEEEECCEEEEECCHHCCCCEEEEEECCHHHHHHHHHHHHHCCCEEEEECCHHHHHHHHHHHHHHHHHHHHCCCCEEEEECCCCEEEECCCEEECCCCCCCCCEECCCCCCCEECCCCCCEEEEEECCCCCCCCCCCEEEEEEEEEEEEEEEEECEEEEEECCCCCCCCCHHHHHHHEECCEEEEECCCEEEEECCCCCEEEEEEEEEEEEEEECCCCCCCCCCCCCCEECCCCHHHCCCCCCCCCCCCCCCCCCCCCCCCCEEEEEEECCCCEEEECCCCCCCCCCCEEEECCCCCCCCEECCCCCCCCEEEECCCCEECCEEEECCCCCCCCEEEEEECC',
 'seq': 'DSGCVINWKGRELKCGSGIFVTNEVHTWTEQYKFQADSPKRLSAAIGKAWEEGVCGIRSATRLENIMWKQISNELNHILLENDMKFTVVVGDVSGILAQGKKMIRPQPMEHKYSWKSWGKAKIIGADVQNTTFIIDGPNTPECPDNQRAWNIWEVEDYGFGIFTTNIWLKLRDSYTQVCDHRLMSAAIKDSKAVHADMGYWIESEKNETWKLARASFIEVKTCIWPKSHTLWSNGVLESEMIIPKIYGGPISQHNYRPGYF

In [121]:
def avaliar_epitopos_com_horiz(df_epitopos, conf, pred, seq):
    """Cruza o DataFrame de epítopos com as strings de predição do .horiz já processadas."""
    resultados = []
    
    for idx, row in df_epitopos.iterrows():
        # Identifica a coluna correta do peptídeo dependendo do estágio do pipeline
        col_pep = 'peptide' if 'peptide' in df_epitopos.columns else 'Epitope sequence'
        peptideo = row[col_pep]
        start = int(row['start'])
        end = int(row['end'])
        
        # Ajuste para índice em Python (base 0)
        idx_inicio = start - 1
        idx_fim = end
        
        # Extrai os trechos correspondentes nas strings do horiz
        sub_pred = pred[idx_inicio:idx_fim]
        sub_conf = conf[idx_inicio:idx_fim]
        
        # Conta a proporção de cada estrutura secundária
        # 'H' = Hélice, 'E' = Fita Beta, 'C' ou '.' = Coil/Alça
        n_coils = sub_pred.count('C') + sub_pred.count('.')
        n_helix = sub_pred.count('H')
        n_strand = sub_pred.count('E')
        
        resultados.append({
            'Epitope': peptideo,
            'Start': start,
            'End': end,
            'Pred_SS': sub_pred,
            'Coils': n_coils,
            'Helices': n_helix,
            'Strands': n_strand
        })
        
    return pd.DataFrame(resultados)

In [122]:
conf = denv_horiz["conf"]
pred = denv_horiz["pred"]
seq = denv_horiz["seq"]

avaliacao_epitopos = avaliar_epitopos_com_horiz(conservacy_result_filtered, conf, pred, seq)
avaliacao_epitopos

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands
0,HTWTEQYKF,26,34,HCCCCEEEE,4,1,4


In [123]:
# Calcula o percentual de Coils (flexibilidade) para cada epítopo
avaliacao_epitopos['Percent_Coils'] = (
    avaliacao_epitopos['Coils'] / (avaliacao_epitopos['Coils'] + avaliacao_epitopos['Helices'] + avaliacao_epitopos['Strands'])
) * 100

# Ordena os epítopos do mais flexível (maior percentual de coils) para o mais rígido
epitopos_flexiveis = avaliacao_epitopos.sort_values(
    by=['Percent_Coils', 'Coils'], 
    ascending=[False, False]
).reset_index(drop=True)

epitopos_flexiveis

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,HTWTEQYKF,26,34,HCCCCEEEE,4,1,4,44.444444


#### Filtrar epítopos que tenham pela % de sua estrutura composta por Coils

In [124]:
valor_corte_flexibilidade = parametros.corte_flexibilidade
print(f"Valor de corte de flexibilidade (Percent_Coils): {valor_corte_flexibilidade}%")

Valor de corte de flexibilidade (Percent_Coils): 35.0%


In [125]:
corte_flexibilidade = valor_corte_flexibilidade
epitopos_filtrados_flex = epitopos_flexiveis[epitopos_flexiveis['Percent_Coils'] >= corte_flexibilidade]

epitopos_filtrados_flex

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,HTWTEQYKF,26,34,HCCCCEEEE,4,1,4,44.444444


### Merge de dataframes

In [126]:
# Seleciona apenas as colunas exclusivas de conservação (excluindo 'start' e 'end' repetidos, se houver)
colunas_para_pegar = [c for c in conservacy_result_filtered.columns if c not in ['start', 'end', 'Epitope sequence']]
colunas_para_pegar.append('Epitope sequence') # Garante que a chave está incluída

# Realiza o merge limpo
df_consolidado = pd.merge(
    epitopos_filtrados_flex,
    conservacy_result_filtered[colunas_para_pegar],
    left_on='Epitope',
    right_on='Epitope sequence',
    how='inner'
)

# Remove se sobrou a coluna 'Epitope sequence' duplicada:
if 'Epitope sequence' in df_consolidado.columns and 'Epitope' in df_consolidado.columns:
    df_consolidado = df_consolidado.drop(columns=['Epitope sequence'])

df_consolidado

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils,Epitope name,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,alelos,percent_match
0,HTWTEQYKF,26,34,HCCCCEEEE,4,1,4,44.444444,NP 1,9,99.80% (1013/1015),88.89%,100.00%,10,0.28,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",99.8


In [127]:
epitopos_filtrados_flex.to_csv("epitopos_validados_flexibilidade_tcell1.csv", index=False)